In [1]:
#| default_exp large

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [4]:
#| export
from os import getenv
model_path = getenv("MODEL")
from rest.gen import generate


In [5]:
model_path = 'large/pelevin'

In [6]:
#| export
seq_length = 1024

full_path = f'./models/{model_path}'
import torch
from transformers import GPT2LMHeadModel,GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(full_path, pad_token_id = 50256)
model = GPT2LMHeadModel.from_pretrained(full_path, torch_dtype=torch.bfloat16)
model.config.pad_token_id = model.config.eos_token_id
model.cuda()
model.eval();

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/_utils.py:835: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [7]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=5120, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=5120)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

In [8]:
sum(p.numel() for p in model.parameters())

774030080

In [9]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool, temperature:float=1.0):
    return generate(model, tokenizer, seq_length, prompt, length, num_samples, allow_linebreak, temperature)

In [10]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:649: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[[63], [12], [331], [14], [3646], [98], [67], [64], [203, 203], [176, 228, 237], [176, 124, 128], [311, 106], [635], [1011], [6610], [17343], [33162], [5486], [13092], [23669], [47006], [2414], [4218], [7610], [15680], [37508], [6730], [13373], [24529], [47061], [1015], [6162], [2460], [18293], [40320], [4990], [10367], [27484], [3559], [7401], [36139], [203], [692]]
CPU times: user 3.06 s, sys: 99.6 ms, total: 3.16 s
Wall time: 2.79 s


[' вы – первый и последний Достоевский, – ответил Игорь Михайлович. – Сначала вы задумали поместить нас в сортир, потом в нары, теперь в гаражи.',
 ' – резал матрац себе под старость. Только не учел одного обстоятельства – размаха советской (вернее, российской) психиатрии. Про него ни Гомер, ни Пушкин не знали, и этим я горжусь.',
 ' полный дебил. Я же тебе как человеку говорю. Ты зачем с собой такие провода таскаешь?',
 ' – настоящий Колчак» — так скажет он в «Льве Толстом», и я знаю, что это не просто слова, – это развернутая программа действий, которая точными выстрелами будет до конца служить идеалам его революции, и мы все']